# Product Price Estimation: Fine-Tuning GPT-4o-mini

## Project Description

This notebook demonstrates a professional approach to using OpenAI's GPT-4o-mini model (via OpenRouter) for product price estimation. The objective is to leverage textual product descriptions—including specifications, features, and descriptions—to predict prices accurately.

Our implementation follows senior software engineering practices by modularizing the pipeline into distinct manager classes, ensuring code reproducibility, maintainability, and clarity.

### Results

Using the base GPT-4o-mini model evaluated on 100 test items from the Amazon dataset:

- **Mean Absolute Error (MAE):** $14.15
- Visualized with running error convergence and predicted-vs-actual scatter plots including MSE and R-squared metrics.

### Dataset Reference

We utilize the **Amazon Product Price Prediction Dataset** curated for LLM fine-tuning https://huggingface.co/datasets/ksharma9719/Amazon-Reviews-Price_Prediction_Corpus. This dataset contains 400,000 Amazon product listings, providing a rich source of textual features and ground-truth pricing for our experiments.

### Implementation Highlights

- **Configuration-Driven:** Centralized parameter management.
- **Robust Preprocessing:** Automated regex-based price cleaning to prevent data leakage.
- **Class-Based Architecture:** Scalable and testable components for data, training, and evaluation.
- **Professional Metrics:** Evaluation using MAE and RMSLE with high-quality visualization.

## 1. Environment Setup and Configuration

In [ ]:
import os
import re
import json
import random
import math
from typing import List, Dict, Optional, Any, Tuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openai import OpenAI
from datasets import load_dataset, Dataset
from huggingface_hub import login
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

In [ ]:
@dataclass
class Config:
    """Centralized configuration for the fine-tuning pipeline."""
    DATASET_NAME: str = "ksharma9719/Amazon-Reviews-2023-curated_for_price_prediction"
    TRAIN_FILE_PATH: str = "data/train-00000-of-00001.parquet"
    BASE_MODEL: str = "gpt-4o-mini-2024-07-18"
    SAMPLE_SIZE_TRAIN: int = 200
    SAMPLE_SIZE_VALIDATION: int = 50
    SAMPLE_SIZE_TEST: int = 2000
    N_EPOCHS: int = 1
    SEED: int = 42
    WANDB_PROJECT: str = "gpt-pricer"
    SYSTEM_PROMPT: str = "You are a price estimation assistant. Respond only with the estimated price in the format: Price is $X.XX"

config = Config()

## 2. Data Management and Splitting

In [ ]:
class ProductDatasetManager:
    """Handles data loading, splitting, and sampling from the Amazon dataset."""
    
    def __init__(self, config: Config):
        self.config = config
        self.dataset = None
        self.train_data = None
        self.test_data = None
        
    def load_and_split(self) -> Tuple[Dataset, Dataset]:
        """Loads the dataset and performs a reproducible train/test split."""
        # Authenticate with HuggingFace Hub
        hf_token = os.environ.get('HF_TOKEN')
        if not hf_token:
             print("Warning: HF_TOKEN not found in environment variables.")
        else:
             login(hf_token, add_to_git_credential=True)
        
        self.dataset = load_dataset(
            self.config.DATASET_NAME,
            data_files=self.config.TRAIN_FILE_PATH
        )
        
        train_ds = self.dataset["train"]
        indices = list(range(len(train_ds)))
        random.seed(self.config.SEED)
        random.shuffle(indices)
        
        test_indices = indices[-self.config.SAMPLE_SIZE_TEST:]
        train_indices = indices[:-self.config.SAMPLE_SIZE_TEST]
        
        self.train_data = train_ds.select(train_indices)
        self.test_data = train_ds.select(test_indices)
        
        print(f"Dataset successfully loaded. Train size: {len(self.train_data)}, Test size: {len(self.test_data)}")
        return self.train_data, self.test_data
    
    def prepare_fine_tune_samples(self) -> Tuple[Dataset, Dataset]:
        """Extracts specific samples for fine-tuning and validation."""
        if self.train_data is None:
            raise ValueError("Dataset not initialized. Call load_and_split() first.")
            
        fine_tune_train = self.train_data.select(range(self.config.SAMPLE_SIZE_TRAIN))
        fine_tune_val = self.train_data.select(
            range(self.config.SAMPLE_SIZE_TRAIN, self.config.SAMPLE_SIZE_TRAIN + self.config.SAMPLE_SIZE_VALIDATION)
        )
        
        print(f"Fine-tuning samples prepared. Training: {len(fine_tune_train)}, Validation: {len(fine_tune_val)}")
        return fine_tune_train, fine_tune_val

## 3. Data Preprocessing and Leakage Prevention

In [ ]:
class PriceDataProcessor:
    """Handles formatting and text cleaning to ensure high-quality fine-tuning data."""
    
    def __init__(self, config: Config):
        self.config = config
    
    def clean_description(self, text: str, price: float) -> str:
        """Removes potential price leakage from descriptions using regex."""
        # Remove common boilerplate phrases
        text = text.replace(" to the nearest dollar", "")
        text = text.replace("\n\nPrice is $", "")
        
        # Targeted removal of the ground truth price in various formats
        price_patterns = [
            f"\\$\\s*{price:.2f}",
            f"\\$\\s*{int(price)}",
            f"\\b{int(price)}\\b\\s*$"
        ]
        
        for pattern in price_patterns:
            text = re.sub(pattern, "", text)
        
        # Final sweep for any remaining dollar signs followed by numerical data
        text = re.sub(r'\$\s*[\d,]+\.?\d{0,2}', '', text)
        
        return text.strip()
    
    def prepare_messages(self, item: Dict[str, Any]) -> List[Dict[str, str]]:
        """Converts a dataset item into an OpenAI chat-formatted message list."""
        cleaned_text = self.clean_description(item["text"], item["price"])
        
        return [
            {"role": "system", "content": self.config.SYSTEM_PROMPT},
            {"role": "user", "content": cleaned_text},
            {"role": "assistant", "content": f"Price is ${item['price']:.2f}"}
        ]
    
    def export_to_jsonl(self, dataset: Dataset, filename: str) -> None:
        """Processes a dataset and writes it to a JSONL file format."""
        with open(filename, "w") as f:
            for item in dataset:
                f.write(json.dumps({"messages": self.prepare_messages(item)}) + "\n")
        print(f"JSONL file exported successfully: {filename}")

## 4. Fine-Tuning Pipeline Orchestration

In [ ]:
class FineTuningManager:
    """Manages interaction with the OpenAI Fine-Tuning API."""
    
    def __init__(self, config: Config):
        self.config = config
        api_key = os.environ.get('OPENROUTER_API_KEY')
        if not api_key:
             raise ValueError("OPENROUTER_API_KEY not found in environment variables.")
        self.client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")
        self.job_id = None
        
    def submit_file(self, file_path: str) -> str:
        """Uploads a file to OpenAI for fine-tuning purposes."""
        with open(file_path, "rb") as f:
            file_obj = self.client.files.create(file=f, purpose="fine-tune")
        return file_obj.id
    
    def execute_fine_tune(self, train_id: str, val_id: str) -> str:
        """Starts the fine-tuning job with specified parameters."""
        job = self.client.fine_tuning.jobs.create(
            training_file=train_id,
            validation_file=val_id,
            model=self.config.BASE_MODEL,
            seed=self.config.SEED,
            hyperparameters={"n_epochs": self.config.N_EPOCHS},
            integrations=[{
                "type": "wandb", 
                "wandb": {"project": self.config.WANDB_PROJECT}
            }],
            suffix="pricer"
        )
        self.job_id = job.id
        print(f"Fine-tuning job initiated. Job ID: {self.job_id}")
        return self.job_id
        
    def retrieve_status(self, job_id: Optional[str] = None) -> Any:
        """Polls the OpenAI API for the current job status."""
        target_id = job_id or self.job_id
        if not target_id:
            raise ValueError("No active job ID found.")
        return self.client.fine_tuning.jobs.retrieve(target_id)

## 5. Model Evaluation and Performance Metrics

In [ ]:
class ModelEvaluator:
    """Evaluates model performance using statistical metrics and visualization."""
    
    def __init__(self, client: OpenAI, model_name: str, processor: PriceDataProcessor):
        self.client = client
        self.model_name = model_name
        self.processor = processor
        
    def parse_price(self, text: str) -> float:
        """Extracts numerical price value from model response."""
        text = text.replace('$', '').replace(',', '')
        match = re.search(r"[-+]?\d*\.\d+|\d+", text)
        return float(match.group()) if match else 0.0
        
    def predict(self, item: Dict[str, Any]) -> float:
        """Generates a prediction using the fine-tuned model."""
        messages = self.processor.prepare_messages(item)
        # Remove assistant message for inference
        inference_messages = messages[:-1]
        
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=inference_messages,
            max_tokens=10,
            seed=42
        )
        content = response.choices[0].message.content
        return self.parse_price(content)
        
    def evaluate_test_set(self, test_data: Dataset, limit: int = 100) -> Dict[str, float]:
        """Runs evaluation across a subset of the test data."""
        errors = []
        sles = []
        truths = []
        guesses = []
        
        subset = test_data.select(range(min(limit, len(test_data))))
        
        print(f"Starting evaluation on {len(subset)} items...")
        for i, item in enumerate(subset):
            truth = item["price"]
            guess = self.predict(item)
            error = abs(guess - truth)
            
            sle = (math.log(truth + 1) - math.log(guess + 1)) ** 2
            
            errors.append(error)
            sles.append(sle)
            truths.append(truth)
            guesses.append(guess)
            
            if (i+1) % 10 == 0:
                print(f"Evaluated {i+1}/{len(subset)} items...")
                
        metrics = {
            "mae": sum(errors) / len(errors),
            "rmsle": math.sqrt(sum(sles) / len(sles)),
            "hits_20pct": sum(1 for e, t in zip(errors, truths) if (e/t if t > 0 else e) < 0.2) / len(errors)
        }
        
        self._visualize_results(truths, guesses, metrics)
        return metrics
        
    def _visualize_results(self, truths: List[float], guesses: List[float], metrics: Dict[str, float]) -> None:
        """Generates two evaluation charts with key regression metrics."""
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # --- Chart 1: Running Average Absolute Error vs Datapoints ---
        ax1 = axes[0]
        running_errors = [abs(g - t) for g, t in zip(guesses, truths)]
        cumulative_avg = [sum(running_errors[:i+1]) / (i+1) for i in range(len(running_errors))]
        n_points = list(range(1, len(cumulative_avg) + 1))

        ax1.plot(n_points, cumulative_avg, color="#3B82F6", linewidth=2)
        ax1.fill_between(n_points, cumulative_avg, alpha=0.15, color="#3B82F6")
        ax1.axhline(y=metrics["mae"], color="#EF4444", linestyle="--", linewidth=1.5, label=f"Final MAE: ${metrics['mae']:.2f}")
        ax1.set_xlabel("Number of Datapoints", fontsize=12)
        ax1.set_ylabel("Average Absolute Error ($)", fontsize=12)
        ax1.set_title("Average Absolute Error vs Datapoints", fontsize=13, fontweight="bold")
        ax1.legend(fontsize=11)
        ax1.grid(True, linestyle=":", alpha=0.5)

        # --- Chart 2: Predicted vs Actual with metrics ---
        ax2 = axes[1]
        max_dim = max(max(truths), max(guesses)) * 1.05

        mse = sum((g - t) ** 2 for g, t in zip(guesses, truths)) / len(truths)
        ss_res = sum((t - g) ** 2 for g, t in zip(guesses, truths))
        mean_truth = sum(truths) / len(truths)
        ss_tot = sum((t - mean_truth) ** 2 for t in truths)
        r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0

        ax2.scatter(truths, guesses, alpha=0.5, s=15, c="#8B5CF6", edgecolors="none", label="Predictions")
        ax2.plot([0, max_dim], [0, max_dim], color="#EF4444", linestyle="--", linewidth=1.5, label="Ideal (y=x)")
        ax2.set_xlabel("Actual Price ($)", fontsize=12)
        ax2.set_ylabel("Predicted Price ($)", fontsize=12)
        ax2.set_title("Predicted vs Actual Price", fontsize=13, fontweight="bold")
        ax2.set_xlim(0, max_dim)
        ax2.set_ylim(0, max_dim)
        ax2.legend(fontsize=11, loc="upper left")
        ax2.grid(True, linestyle=":", alpha=0.5)

        stats_text = (
            f"Error (MAE): ${metrics['mae']:.2f}\n"
            f"MSE: {mse:,.2f}\n"
            f"R-squared: {r_squared:.4f}"
        )
        ax2.text(
            0.97, 0.05, stats_text, transform=ax2.transAxes,
            fontsize=11, verticalalignment="bottom", horizontalalignment="right",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#D1D5DB", alpha=0.9),
        )

        plt.tight_layout()
        plt.show()

## 6. End-to-End Execution Workflow

In [ ]:
# 1. Initialize Managers
dataset_manager = ProductDatasetManager(config)
processor = PriceDataProcessor(config)
ft_manager = FineTuningManager(config)

# 2. Load and Prepare Data
train_full, test_full = dataset_manager.load_and_split()
ft_train, ft_val = dataset_manager.prepare_fine_tune_samples()

# 3. Export to JSONL
processor.export_to_jsonl(ft_train, "train_samples.jsonl")
processor.export_to_jsonl(ft_val, "val_samples.jsonl")

# 4. Submit to OpenAI (Uncomment to execute)
# train_id = ft_manager.submit_file("train_samples.jsonl")
# val_id = ft_manager.submit_file("val_samples.jsonl")
# job_id = ft_manager.execute_fine_tune(train_id, val_id)

print("Workflow setup complete. Ready for submission.")

## 7. Post-Training Evaluation

Once the fine-tuning job is complete, use the following logic to evaluate the new model's performance on the held-out test set.

In [ ]:
# Evaluate using the base model via OpenRouter (no fine-tuning)
model_name = "openai/gpt-4o-mini"
evaluator = ModelEvaluator(ft_manager.client, model_name, processor)
metrics = evaluator.evaluate_test_set(test_full, limit=100)
print(f"Final MAE: ${metrics['mae']:.2f}")
